> **Run this in Amazon SageMaker AI.** These notebooks are designed to run in Amazon SageMaker AI Studio. Running them locally (for example in VS Code) or in another environment is not supported and will fail. Complete the [Environment Setup: Amazon SageMaker AI](https://neo4j-partners.github.io/neo4j-sec-filings-graphrag-workshop/workshop/neo4j-sec-filings-graphrag-workshop/1.0/part2-setup-instructions.html) steps first to launch SageMaker AI Studio, then open the notebooks there.

# Hybrid Retriever

Vector search matches by meaning; full-text search matches by keyword. Each misses what the other catches — embeddings blur exact names, tickers, and acronyms, while keyword search misses paraphrase. **Hybrid search** runs both signals over the same chunks and re-ranks the merged results, so a single query benefits from both.

**Learning Objectives:**
- Create a full-text index on chunk text and run keyword search
- Use `HybridRetriever` to fuse vector and full-text results
- Compare the `NAIVE` and `LINEAR` re-rankers and tune the `alpha` weight
- Use `HybridCypherRetriever` to add graph traversal to hybrid search

The two signals use different score scales (cosine similarity is roughly 0 to 1; Lucene relevance is unbounded), so the retriever normalizes each onto a common 0-to-1 scale before merging. That normalization and blend is the re-ranking step.

In [ ]:
%pip install "neo4j-graphrag[bedrock]>=1.18.0" -q

In [ ]:
import os

import neo4j
from lib.data_utils import get_embedder, get_llm
from neo4j_graphrag.retrievers import VectorRetriever, HybridRetriever, HybridCypherRetriever
from neo4j_graphrag.types import HybridSearchRanker, RetrieverResultItem
from neo4j_graphrag.generation import GraphRAG
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

# notifications_min_severity='OFF' silences the per-query deprecation warning
# Aura now emits for db.index.vector.queryNodes, which the retrievers call.
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
    notifications_min_severity='OFF',
)
driver.verify_connectivity()

embedder = get_embedder()
llm = get_llm()

print('Connected to Neo4j!')
print(f'LLM: {llm.model_name}')
print(f'Embedder: {embedder.model_id}')

## Create the Full-text Index

Hybrid search needs a full-text index over chunk text. The Lab 1 seed load created the `chunkEmbeddings` vector index and the `search_entities` full-text index (on entity names), but not a full-text index on `Chunk.text`. Create `search_chunks` now.

`IF NOT EXISTS` makes this safe to re-run, and `db.awaitIndexes` blocks until the index is online so the searches below don't race the index build.

In [ ]:
driver.execute_query(
    "CREATE FULLTEXT INDEX search_chunks IF NOT EXISTS FOR (c:Chunk) ON EACH [c.text]"
)
driver.execute_query("CALL db.awaitIndexes(300)")

print("Full-text index 'search_chunks' is ready.")

## Full-text Keyword Search

Full-text search ranks chunks by Lucene relevance for the given keywords. Unlike vector search, it matches terms *literally*, which makes it strong on exact names, tickers, and model numbers. Search for `H100` — a specific NVIDIA data center GPU — directly against the `search_chunks` index.

In [ ]:
query = "H100"

records, _, _ = driver.execute_query(
    """
    CALL db.index.fulltext.queryNodes('search_chunks', $query)
    YIELD node, score
    RETURN node.text AS text, score
    ORDER BY score DESC
    LIMIT 5
    """,
    query=query,
)

print(f'Keyword query: "{query}"')
print(f'Results returned: {len(records)}\n')
for i, record in enumerate(records, 1):
    print(f'{i}. Score: {record["score"]:.4f}')
    print(f'   {record["text"][:150]}...\n')

assert records, "Expected full-text matches for the query"

## Vector vs Full-text: The Exact-Token Gap

Before fusing them, see *why* you want both. `H100` is a specific NVIDIA chip. Vector search matches by *meaning*, so it drifts to general passages about GPUs and data centers and can miss the literal model name. Full-text matches the token *exactly*.

Compare how many of the top five chunks from each strategy actually contain `H100`.

In [ ]:
# The exact-token gap: how many of the top-5 chunks from each strategy contain "H100"?
token = "H100"
query = "H100"

vector_retriever = VectorRetriever(
    driver=driver,
    index_name='chunkEmbeddings',
    embedder=embedder,
    return_properties=['text'],
)

vector_result = vector_retriever.search(query_text=query, top_k=5)
vector_texts = [str(item.content) for item in vector_result.items]

fulltext_records, _, _ = driver.execute_query(
    """
    CALL db.index.fulltext.queryNodes('search_chunks', $query)
    YIELD node, score
    RETURN node.text AS text
    ORDER BY score DESC
    LIMIT 5
    """,
    query=query,
)
fulltext_texts = [record["text"] for record in fulltext_records]


def count_token(texts, term):
    return sum(1 for text in texts if term.lower() in text.lower())


vector_hits = count_token(vector_texts, token)
fulltext_hits = count_token(fulltext_texts, token)

print(f'Query: "{query}"\n')
print(f'Vector search    : "{token}" appears in {vector_hits}/{len(vector_texts)} chunks')
print(f'Full-text search : "{token}" appears in {fulltext_hits}/{len(fulltext_texts)} chunks')
print()
print('Vector search matches on meaning, so it drifts to general GPU and data-center')
print('passages and can miss the exact model name. Full-text matches the token literally.')
print('Hybrid search (next) keeps both signals, so neither miss goes unrecovered.')

assert fulltext_hits >= vector_hits, "Full-text should find the exact token at least as often as vector"

## Hybrid Retriever

`HybridRetriever` embeds the query for the vector arm and passes the same text as keywords to the full-text arm, then merges and re-ranks the two result sets. It takes both index names plus the embedder, and otherwise behaves like the `VectorRetriever` from Notebook 01.

In [ ]:
hybrid_retriever = HybridRetriever(
    driver=driver,
    vector_index_name='chunkEmbeddings',
    fulltext_index_name='search_chunks',
    embedder=embedder,
    return_properties=['text'],
)

query = "risks to H100 and A100 sales"
result = hybrid_retriever.search(query_text=query, top_k=5)

print(f'Query: "{query}"')
print(f'Results returned: {len(result.items)}\n')
for i, item in enumerate(result.items, 1):
    score = item.metadata.get('score', 0.0)
    print(f'{i}. Score: {score:.4f}')
    print(f'   {str(item.content)[:150]}...\n')

assert result.items, "Expected hybrid search results"

## Re-ranking: NAIVE vs LINEAR

Because the two arms report scores on different scales, the retriever normalizes each to 0-to-1 before combining them. The `ranker` controls how they combine:

- **`NAIVE`** (default): keeps the maximum normalized score per chunk across the two arms.
- **`LINEAR`**: a weighted blend, `alpha * vector + (1 - alpha) * fulltext`. Raise `alpha` toward `1.0` to favor semantic meaning; lower it toward `0.0` to favor exact keywords. `alpha` defaults to `0.5`.

Run the same query three ways and watch the exact `H100` / `A100` chunks move in and out of the top five as `alpha` shifts: keyword-weighted ranking surfaces them, meaning-weighted ranking drops them.

In [ ]:
query = "risks to H100 and A100 sales"

def show(label, items):
    hits = sum(1 for item in items if 'h100' in str(item.content).lower())
    print(f'--- {label}  ("H100" in {hits}/{len(items)} chunks) ---')
    for i, item in enumerate(items, 1):
        score = item.metadata.get('score', 0.0)
        print(f'{i}. {score:.4f}  {str(item.content)[:90]}...')
    print()

naive = hybrid_retriever.search(query_text=query, top_k=5, ranker=HybridSearchRanker.NAIVE)
meaning = hybrid_retriever.search(query_text=query, top_k=5, ranker=HybridSearchRanker.LINEAR, alpha=0.9)
keywords = hybrid_retriever.search(query_text=query, top_k=5, ranker=HybridSearchRanker.LINEAR, alpha=0.1)

show('NAIVE (max of both signals)', naive.items)
show('LINEAR alpha=0.9 (favor meaning)', meaning.items)
show('LINEAR alpha=0.1 (favor keywords)', keywords.items)

## GraphRAG with Hybrid Retrieval

Drop the hybrid retriever into a `GraphRAG` pipeline. The LLM now answers from context that was retrieved by both semantic and keyword matching. Ranker options pass through `retriever_config`.

In [ ]:
rag = GraphRAG(llm=llm, retriever=hybrid_retriever)

query = "How do U.S. export controls affect NVIDIA's H100 sales?"
response = rag.search(
    query,
    retriever_config={'top_k': 5, 'ranker': HybridSearchRanker.LINEAR, 'alpha': 0.5},
    return_context=True,
)

print(f'Query: "{query}"\n')
print('Answer:')
print(response.answer)

print('\n\n=== Retrieved Context ===')
for i, item in enumerate(response.retriever_result.items, 1):
    score = item.metadata.get('score', 0.0)
    content_str = str(item.content)
    preview = content_str[:200] + '...' if len(content_str) > 200 else content_str
    print(f'\n[{i}] Score: {score:.4f}')
    print(f'    {preview}')

## Hybrid + Graph Traversal

`HybridCypherRetriever` combines hybrid search with the Cypher graph traversal from Notebook 02. Hybrid search finds the chunks; the retrieval query then enriches each match with its filing, company, products, and risk factors. The retrieval query is identical to the one used with `VectorCypherRetriever` — only the retrieval strategy that feeds it changes.

In [ ]:
RETRIEVAL_QUERY = """
MATCH (node)-[:FROM_DOCUMENT]->(doc:Document)
OPTIONAL MATCH (doc)<-[:FILED]-(company:Company)
WITH node, doc, score, company
RETURN node.text AS text,
       score,
       {document: doc.accessionNumber,
        filingType: doc.filingType,
        company: company.name,
        products: collect { MATCH (p:Product)-[:FROM_CHUNK]->(node) RETURN p.name },
        risks: collect { MATCH (r:RiskFactor)-[:FROM_CHUNK]->(node) RETURN r.name }
       } AS metadata
"""

def format_record(record: neo4j.Record) -> RetrieverResultItem:
    """Separate chunk text (content for the LLM) from structured graph metadata."""
    metadata = record.get("metadata") or {}
    metadata["score"] = record.get("score")
    return RetrieverResultItem(content=record.get("text", ""), metadata=metadata)

hybrid_cypher_retriever = HybridCypherRetriever(
    driver=driver,
    vector_index_name='chunkEmbeddings',
    fulltext_index_name='search_chunks',
    retrieval_query=RETRIEVAL_QUERY,
    embedder=embedder,
    result_formatter=format_record,
)

query = "risks to NVIDIA's H100 and A100 chips"
result = hybrid_cypher_retriever.search(
    query_text=query, top_k=3, ranker=HybridSearchRanker.LINEAR, alpha=0.3
)

print(f'Query: "{query}"\n')
for i, item in enumerate(result.items, 1):
    meta = item.metadata or {}
    print(f'[{i}] Score: {meta.get("score", 0):.4f} | Company: {meta.get("company", "N/A")}')
    print(f'    Products: {meta.get("products", [])}')
    print(f'    Risks: {meta.get("risks", [])}')
    print(f'    Text: {str(item.content)[:200]}...\n')

assert result.items, "Expected hybrid + cypher results"

## Summary

You added keyword and hybrid retrieval to the GraphRAG toolkit:

| Approach | Matches by | Re-ranking |
|----------|-----------|-----------|
| **Full-text search** | Keywords (Lucene) | Lucene relevance only |
| **HybridRetriever** | Vector + keywords | `NAIVE` (max) or `LINEAR` (alpha blend) |
| **HybridCypherRetriever** | Vector + keywords, plus graph traversal | Same rankers, then enriched with graph context |

Hybrid search normalizes the two score scales and merges them, so exact names and tickers (keyword) and paraphrased meaning (vector) both contribute to a single ranked result. The `alpha` weight tunes that balance per query, and `HybridCypherRetriever` layers the same graph traversal you built in Notebook 02 on top.

---

**Next:** [Strands GraphRAG Agent](../Lab_4_GraphRAG_Agent/01_strands_graphrag_agent.ipynb) — let an agent choose between these retrieval strategies

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')